[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ksankaran/hello-model/blob/main/hello_model.ipynb)

# Hello, Model!

You already know what a model is - you just don't know you know.

Ever drawn a trendline in a spreadsheet? That's a model.  
Ever seen your phone predict the next word you'll type? That's a model.  
Ever set a thermostat to 72°F? That's a model deciding when to turn on the heat.

A **model** is just a function with **knobs** (we call them *parameters*).  
You give it an input, it gives you an output.  
**Training** is the process of turning those knobs until the outputs are good.

Let's build one from scratch. No libraries. No magic. Just Python.

## Step 1: The Data

We have 7 houses. We know each house's square footage and its sale price.

| Sq Ft | Price ($1000s) |
|-------|----------------|
| 600   | 150            |
| 800   | 200            |
| 1000  | 250            |
| 1200  | 280            |
| 1500  | 350            |
| 1800  | 400            |
| 2200  | 500            |

**Our goal:** Given a new house's square footage, predict its price.

Let's put this data in Python.

In [ ]:
# Our data - 7 houses
sqft  = [600, 800, 1000, 1200, 1500, 1800, 2200]
price = [150, 200,  250,  280,  350,  400,  500]   # in $1000s

print(f"We have {len(sqft)} houses.")
print(f"Smallest: {sqft[0]} sqft -> ${price[0]}k")
print(f"Largest:  {sqft[-1]} sqft -> ${price[-1]}k")

## Step 2: The Model (Just a Line)

Remember from school: **y = mx + b**

- `x` = square footage (input)
- `y` = predicted price (output)
- `m` = slope (knob #1) - how much price increases per sqft
- `b` = intercept (knob #2) - base price

That's our **model**. Two knobs: `m` and `b`.

Let's start with random values and see what happens.

In [ ]:
# Our model: y = m * x + b
# Let's start with random guesses for m and b
m = 0.05   # guess: 5 cents per sqft?
b = 10.0   # guess: base price $10k?

def predict(x):
    """Our model - a simple line."""
    return m * x + b

# Let's see how our random guesses do
print("Sqft  | Actual | Predicted | Off by")
print("------|--------|-----------|-------")
for x, actual in zip(sqft, price):
    pred = predict(x)
    print(f"{x:5d} | ${actual:5d}k | ${pred:8.1f}k | ${abs(actual - pred):6.1f}k")

Those predictions are **terrible**. The knobs (`m` and `b`) are set wrong.

But how wrong? We need a way to measure "wrongness." That's called a **loss function**.

## Step 3: Measuring Wrongness (The Loss Function)

For each house, our prediction is off by some amount. We want a single number
that captures **how wrong we are overall**.

The standard approach: **Mean Squared Error (MSE)**

1. For each house: `error = predicted - actual`
2. Square it: `error²` (makes all errors positive, penalizes big errors more)
3. Average all the squared errors

Lower MSE = better model. MSE of 0 = perfect predictions.

In [ ]:
def compute_loss(m, b):
    """Mean Squared Error - how wrong are we overall?"""
    total_error = 0
    for x, actual in zip(sqft, price):
        predicted = m * x + b
        error = predicted - actual
        total_error += error ** 2       # square it
    return total_error / len(sqft)      # average

loss = compute_loss(m, b)
print(f"With m={m}, b={b}:")
print(f"Loss (MSE) = {loss:,.1f}")
print(f"\nThat's a big number. Our model is very wrong.")

## Step 4: Turning the Knobs by Hand

Before we automate anything, let's build intuition.

Our predictions were too low. What if we increase `m` (price per sqft)?

In [ ]:
# Let's try a few different values of m and see how the loss changes
print("  m    |   b   |    Loss (MSE)")
print("-------|-------|---------------")
for test_m in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    loss = compute_loss(test_m, b=10)
    print(f" {test_m:.2f}  | 10.00 | {loss:>12,.1f}")

print("\nLoss goes DOWN as m increases... up to a point.")
print("Somewhere around m=0.20-0.25, the loss is lowest.")
print("Go past that and the loss goes back UP - we're overshooting.")

We just did what training does, but by hand:
1. Try a value
2. Check the loss
3. Adjust the knob
4. Repeat

But trying every possible value is slow. There's a smarter way: **gradient descent**.

## Step 5: Gradient Descent (The Smart Way to Turn Knobs)

Imagine you're blindfolded on a hilly landscape. You want to reach the lowest valley.
What do you do? **Feel which way is downhill, and take a step in that direction.**

That's gradient descent:
- The "landscape" is the loss for every possible `(m, b)` combination
- The "slope of the ground" is the **gradient** - it tells us which direction increases the loss
- We step in the **opposite** direction (downhill) to reduce the loss

The math for our model `y = mx + b`:

```
gradient_m = average of  2 * (predicted - actual) * x
gradient_b = average of  2 * (predicted - actual)
```

Don't worry about where these formulas come from - it's just calculus applied to MSE.
The important thing: **the gradient points uphill, so we go the other way.**

In [ ]:
def compute_gradients(m, b):
    """Which direction should we nudge m and b to reduce the loss?"""
    grad_m = 0
    grad_b = 0
    n = len(sqft)
    
    for x, actual in zip(sqft, price):
        predicted = m * x + b
        error = predicted - actual
        grad_m += (2/n) * error * x    # how much m contributes to the error
        grad_b += (2/n) * error         # how much b contributes to the error
    
    return grad_m, grad_b

# Let's see the gradients at our starting point
gm, gb = compute_gradients(m=0.05, b=10)
print(f"Gradient for m: {gm:,.1f}")
print(f"Gradient for b: {gb:.1f}")
print(f"\nBoth are negative -> m and b should INCREASE to reduce loss.")
print(f"(Negative gradient = we're on the left side of the valley = move right)")

## Step 6: Training - Let the Computer Turn the Knobs

Now we automate it. On each step:
1. Compute the gradients (which way is downhill?)
2. Nudge `m` and `b` in that direction
3. Repeat

The **learning rate** controls how big each step is. Too big = overshoot. Too small = too slow.

In [ ]:
# --- THIS IS THE ENTIRE TRAINING LOOP ---

m = 0.05          # starting guess
b = 10.0          # starting guess
learning_rate = 0.0000003   # small steps (our x values are big numbers)
epochs = 500      # number of times we adjust the knobs

history = []      # track progress

for epoch in range(epochs):
    # 1. How wrong are we?
    loss = compute_loss(m, b)
    
    # 2. Which way is downhill?
    grad_m, grad_b = compute_gradients(m, b)
    
    # 3. Take a step downhill
    m = m - learning_rate * grad_m
    b = b - learning_rate * grad_b
    
    history.append((epoch, loss, m, b))
    
    # Print progress every 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss:10.1f} | m: {m:.4f} | b: {b:.2f}")

print(f"\n--- Training complete ---")
print(f"Final m = {m:.4f} (price per sqft: ${m*1000:.0f})")
print(f"Final b = {b:.2f} (base price: ${b*1000:,.0f})")
print(f"Final loss = {compute_loss(m, b):.1f}")

## Step 7: Did It Work?

Let's compare our model's predictions to the actual prices.

In [ ]:
print("Sqft  | Actual | Predicted | Off by")
print("------|--------|-----------|-------")
for x, actual in zip(sqft, price):
    pred = m * x + b
    diff = abs(actual - pred)
    print(f"{x:5d} | ${actual:5d}k | ${pred:8.1f}k | ${diff:5.1f}k")

print(f"\nMuch better! Most predictions are within $10-15k.")

## Step 8: The Real Test - Predict a House We've Never Seen

The whole point of a model: **predict on new, unseen data.**

In [ ]:
# Predict prices for houses NOT in our training data
new_houses = [700, 1100, 1600, 2000, 2500]

print("Predicting prices for new houses:\n")
for x in new_houses:
    predicted_price = m * x + b
    print(f"  {x:,} sqft -> ${predicted_price:,.0f}k (${predicted_price*1000:,.0f})")

## Step 9: Visualize It

Let's see our data points and the line our model learned.

In [ ]:
# Simple ASCII plot - no matplotlib needed!

def ascii_plot(sqft, price, m, b):
    """Draw an ASCII scatter plot with the fitted line."""
    width = 60
    height = 20
    
    min_x, max_x = min(sqft) - 100, max(sqft) + 100
    min_y, max_y = 100, 550
    
    grid = [[' ' for _ in range(width)] for _ in range(height)]
    
    def to_grid(x, y):
        col = int((x - min_x) / (max_x - min_x) * (width - 1))
        row = int((1 - (y - min_y) / (max_y - min_y)) * (height - 1))
        return max(0, min(height-1, row)), max(0, min(width-1, col))
    
    # Draw the fitted line
    for px in range(width):
        x = min_x + px / (width - 1) * (max_x - min_x)
        y = m * x + b
        row, col = to_grid(x, y)
        if 0 <= row < height and 0 <= col < width:
            grid[row][col] = '-'
    
    # Draw data points (on top of line)
    for x, y in zip(sqft, price):
        row, col = to_grid(x, y)
        grid[row][col] = '*'
    
    # Print
    print(f"  ${max_y}k |")
    for row in grid:
        print(f"        |{''.join(row)}|")
    print(f"  ${min_y}k |{'_' * width}|")
    print(f"         {min_x}{'sqft':>{width-4}}{max_x}")
    print(f"\n  * = actual data    - = our model's line")

ascii_plot(sqft, price, m, b)

## Step 10: Let's Watch It Learn

Here's the loss going down over training - each step, the model gets a little better.

In [ ]:
# ASCII loss curve
def ascii_loss_curve(history):
    """Show how the loss decreased during training."""
    losses = [h[1] for h in history]
    
    # Sample ~20 points
    step = max(1, len(losses) // 20)
    sampled = losses[::step]
    
    max_loss = max(sampled)
    min_loss = min(sampled)
    width = 40
    
    print("Loss over training epochs:\n")
    for i, loss in enumerate(sampled):
        epoch = i * step
        bar_len = int((loss - min_loss) / (max_loss - min_loss + 1) * width)
        print(f"  Epoch {epoch:3d} | {'#' * bar_len} {loss:,.0f}")
    
    print(f"\n  Loss dropped from {losses[0]:,.0f} to {losses[-1]:,.0f}")
    print(f"  That's a {(1 - losses[-1]/losses[0])*100:.0f}% reduction!")

ascii_loss_curve(history)

## What Just Happened?

Let's recap what we built:

| Concept | In our code | In the real world |
|---------|------------|-------------------|
| **Model** | `y = m*x + b` | Neural networks, GPT, etc. |
| **Parameters** | `m` and `b` (2 knobs) | GPT-4 has ~1.8 trillion |
| **Training data** | 7 houses | Billions of text documents |
| **Loss function** | Mean Squared Error | Cross-entropy, RLHF, etc. |
| **Training** | Gradient descent loop | Same idea, just bigger |
| **Prediction** | `predict(1600)` -> `$340k` | "Complete this sentence..." |

**The core idea is identical.** A model is a function with knobs. Training turns the knobs until the outputs match reality. That's it.

The difference between our 2-parameter model and GPT isn't the concept - it's the **scale**:
- More parameters (billions instead of 2)
- More data (internet-scale instead of 7 houses)
- More compute (GPU clusters instead of your laptop)
- Fancier architectures (transformers instead of a line)

But the loop is always the same:
1. Predict
2. Measure how wrong
3. Adjust the knobs
4. Repeat

## Try It Yourself

Things to experiment with:
- Change the `learning_rate` - what happens if it's 10x bigger? 10x smaller?
- Change `epochs` to 50 - does the model learn enough?
- Add more data points to the `sqft` and `price` lists
- What happens if one house is wildly overpriced? (e.g., 1000 sqft -> $900k)